# Tuto how to train a GNN

## First method: Inductive Training

1. Import stuff

In [1]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_inductive import GraphDatasetInductive
from models.model_utils import train_inductive, test_inductive

2. import your model (that you defined in the "/models" file)

In [2]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [3]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 64
out_channels      = 64
num_layers        = 2
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 4
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [4]:
# dataset + train/test split
full_dataset = GraphDatasetInductive(json_dir)
n_train      = int(0.8 * len(full_dataset))
n_test       = len(full_dataset) - n_train
train_ds, test_ds = random_split(full_dataset, [n_train, n_test])

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=batch_size)

/home/shangeeth/wsl_deeplsd_39/lib/python3.9/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


5. Initialize your model

In [5]:
# model, optimizer, loss
model     = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCELoss()

6. Train your model

In [6]:
torch.cuda.empty_cache()
# training loop
for epoch in range(1, epochs+1):

    train_loss = train_inductive(model, train_loader, optimizer, criterion, device=device)


    #train_n_acc, train_e_acc = test_inductive(model, train_loader, device=device)
    test_n_acc,  test_e_acc  = test_inductive(model, test_loader,  device=device)

    print(f'Epoch {epoch:02d} | '
          f'Loss: {train_loss:.4f} | '
          #f'Train Node Acc: {train_n_acc:.4f} | Train Edge Acc: {train_e_acc:.4f} | '
          f'Test Node Acc:  {test_n_acc:.4f} | Test Edge Acc:  {test_e_acc:.4f}')

Loss for mini-batch 0: 1.6278057098388672
Loss for mini-batch 1: 1.241091251373291
Loss for mini-batch 2: 1.2618184089660645
Loss for mini-batch 3: 1.1808396577835083
Loss for mini-batch 4: 1.5158169269561768
Loss for mini-batch 5: 1.8810285329818726
Loss for mini-batch 6: 1.15826416015625
Loss for mini-batch 7: 2.308544635772705
Loss for mini-batch 8: 1.3309450149536133
Loss for mini-batch 9: 1.3572733402252197
Loss for mini-batch 10: 1.3395601511001587
Loss for mini-batch 11: 1.3441193103790283
Loss for mini-batch 12: 1.3048864603042603
Loss for mini-batch 13: 1.3022215366363525
Loss for mini-batch 14: 1.210681676864624
Loss for mini-batch 15: 1.213404893875122
Loss for mini-batch 16: 1.2104346752166748
Loss for mini-batch 17: 1.1644970178604126
Loss for mini-batch 18: 1.1655206680297852
Loss for mini-batch 19: 1.1831517219543457
Correct nodes for mini-batch 0: 629
Correct edges for mini-batch 0: 270424
Correct nodes for mini-batch 1: 1311
Correct edges for mini-batch 1: 514116
Corre

## Second method: Transductive Training

1. Import stuff

In [7]:
import os
import json
import numpy as np
import torch
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
import torch.nn as nn
import torch_geometric.nn as pyg_nn

from models.dataset_transductive import GraphDatasetTransductive # This changed
from models.model_utils import train_combined, test_combined # This changed

2. import your model (that you defined in the "/models" file)

In [8]:
from models.base_gat_combined import GATClassifierCombined


3. Define your parameters

In [9]:
# Paramaeters
json_dir          = './json_output/'
# Model parameters
in_channels       = 1280
hidden_channels   = 256
out_channels      = 256
num_layers        = 5
dropout           = 0.1
act               = 'relu'
# Training Prameters
lr                = 1e-2
weight_decay      = 5e-4
batch_size        = 2
epochs            = 10

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

4. Load data and split it into training/Test dataset

In [10]:
# Initialise dataset and dataloader
dataset = GraphDatasetTransductive(json_dir, struct_thresh=0.6, textural_thresh=0.4)  # This changed

loader = DataLoader(dataset, batch_size=batch_size, shuffle=True) # This changed

5. Initialize your model

In [11]:
# Initialize model
model = GATClassifierCombined(in_channels=in_channels, 
                            hidden_channels=hidden_channels, 
                            out_channels=out_channels, 
                            num_layers=num_layers, 
                            dropout=dropout, 
                            act=act).to(device)

# Loss function and Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = torch.nn.BCELoss()

6. Train your model

In [12]:
# training loop    # This changed
for epoch in range(1, epochs+1):
    loss = train_combined(model, optimizer, criterion, loader, node_loss_weight=1.0, edge_loss_weight=1.0)
    train_acc_struct, test_acc_struct, train_acc_coplanarity, test_acc_coplanarity = test_combined(model, criterion, loader,
                                                                                                   threshold_structural=0.5,
                                                                                                   threshold_coplanarity=0.5)
    
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')
    print(f'Structural/Textural: Train Acc: {train_acc_struct:.4f}, Test Acc: {test_acc_struct:.4f}')
    print(f'Coplanarity: Train Acc: {train_acc_coplanarity:.4f}, Test Acc: {test_acc_coplanarity:.4f}')

ValueError: need at least one array to concatenate